# 04 — Baseline Model Training & Evaluation

**Pipeline position:** Step 4 of 7

**Purpose:** Train and evaluate five baseline classifiers (Logistic Regression,
Random Forest, XGBoost, LightGBM, MLP) with bootstrap 95% CI for AUC-ROC,
AUC-PR, calibration, Decision Curve Analysis, and clinical performance metrics.

**License:** MIT

## Prerequisites
- Run `03_feature_selection.ipynb` first
- Inputs: `data_training_selected.csv`, `data_internal_validation_selected.csv`, `data_temporal_validation_selected.csv`

## Outputs
- `model_results/` directory containing plots and `performance_metrics.csv`


In [ ]:
# SPDX-License-Identifier: MIT
# ============================================================
# Dependency check — run this cell first
# ============================================================
import importlib.util

_required = ['pandas', 'numpy', 'sklearn', 'matplotlib', 'seaborn', 'openpyxl']
_missing = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    raise ImportError(
        f"Missing packages: {_missing}\n"
        f"Install with: pip install {' '.join(_missing)}"
    )
print("Core dependencies satisfied.")

if importlib.util.find_spec('xgboost') is None:
    print("WARNING: 'xgboost' not installed — run: pip install xgboost")
else:
    import xgboost
    print(f"  xgboost {getattr(xgboost, '__version__', 'ok')}")
if importlib.util.find_spec('lightgbm') is None:
    print("WARNING: 'lightgbm' not installed — run: pip install lightgbm")
else:
    import lightgbm
    print(f"  lightgbm {getattr(lightgbm, '__version__', 'ok')}")

In [ ]:
# ============================================================
# Configuration
# ============================================================
OUTCOME_VAR  = "SpontAbortion"
INDEX_VAR    = "Index"
DATE_VAR     = "BaseInfoDate"
RANDOM_STATE = 42
N_BOOTSTRAP  = 100          # bootstrap resamples for 95% CI
TRAIN_FILE   = "data_training_selected.csv"
INTVAL_FILE  = "data_internal_validation_selected.csv"
TEMPVAL_FILE = "data_temporal_validation_selected.csv"
OUTPUT_DIR   = "model_results"
DPI          = 300


In [2]:
"""
Multi-model prediction and performance evaluation script
========================================================
Models: Logistic Regression, Random Forest, XGBoost, GradientBoosting, MLP
Metrics:
  - AUC-ROC (with 95% CI)
  - AUC-PR  (with 95% CI)
  - Calibration plot
  - Decision Curve Analysis (DCA)
  - Confusion matrix
  - Clinical metrics table (Sensitivity, Specificity, PPV, NPV with 95% CI)

Dataset splits: Training / Internal Validation / Temporal Validation

Dependencies:
  pip install pandas numpy scikit-learn xgboost matplotlib seaborn openpyxl
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, accuracy_score, f1_score,
    brier_score_loss, log_loss
)
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
import warnings
import time
import os

warnings.filterwarnings("ignore")

# Attempt to import xgboost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("WARNING: xgboost not installed — XGBoost skipped. Run: pip install xgboost")

# Attempt to import lightgbm
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("WARNING: lightgbm not installed — LightGBM skipped. Run: pip install lightgbm")

# ============================================================
# 1. Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"
RANDOM_STATE = 42
N_BOOTSTRAP = 100           # Bootstrap resamples for 95% CI estimation

# Input files (Boruta-selected features)
TRAIN_FILE = "data_training_selected.csv"
INTVAL_FILE = "data_internal_validation_selected.csv"
TEMPVAL_FILE = "data_temporal_validation_selected.csv"

# Output directory
OUTPUT_DIR = "model_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot configuration
DPI = 300
COLOR_PALETTE = {
    "LR":   "#2E86AB",    # blue
    "RF":   "#5BB37A",    # green
    "XGB":  "#E8637A",    # red
    "LGBM": "#F5A623",    # orange
    "MLP":  "#9B59B6",    # purple
    "GBM":  "#1ABC9C",    # cyan
}
MODEL_NAMES = {
    "LR":   "Logistic Regression",
    "RF":   "Random Forest",
    "XGB":  "XGBoost",
    "LGBM": "LightGBM",
    "MLP":  "MLP",
    "GBM":  "GradientBoosting",
}

# Variable type lists (for encoding)
ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]
NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair", "HusbMental", "HusbIntel",
    "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva", "HusbPubHair", "AdamsApple",
    "Penis", "Foreskin", "Testis", "Epididymis", "VasDeferens",
    "Varicocele", "WifeUrine", "HusbUrine", "WifeHepB", "HusbHepB",
]
BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh", "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM", "SyphilisScr", "HusbSyphilis",
    "GynUltrasound", "HusbDiseaseHis", "HusbBirthDefect",
    "HusbAndrologyDis", "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease", "HusbMeatEgg",
    "HusbVegAverse", "HusbRawMeat", "HusbSmoke", "HusbPassSmoke",
    "HusbAlcohol", "HusbDrugUse", "HusbEnvExpose", "HusbStress",
    "HusbRelatStress", "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]


# ============================================================
# 2. Load and encode data
# ============================================================
print("=" * 70)
print("STEP 1: Load data")
print("=" * 70)

df_train = pd.read_csv(TRAIN_FILE)
df_intval = pd.read_csv(INTVAL_FILE)
df_tempval = pd.read_csv(TEMPVAL_FILE)

print(f"  Training    : {len(df_train):,}")
print(f"  Internal val: {len(df_intval):,}")
print(f"  Temporal val: {len(df_tempval):,}")


def encode_and_prepare(df_train, df_intval, df_tempval):
    """Encode and prepare all three dataset splits"""
    drop_cols = [INDEX_VAR, DATE_VAR, "Dataset"]
    datasets = {}

    for name, df in [("train", df_train), ("intval", df_intval), ("tempval", df_tempval)]:
        df_m = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

        # Outcome encoding
        if df_m[OUTCOME_VAR].dtype == object or df_m[OUTCOME_VAR].dtype == bool:
            df_m[OUTCOME_VAR] = df_m[OUTCOME_VAR].map({
                True: 1, False: 0, "True": 1, "False": 0,
                "TRUE": 1, "FALSE": 0
            })
        datasets[name] = df_m

    # Fit encoders on training set only
    label_encoders = {}
    for col in ORDINAL_VARS + NOMINAL_VARS:
        for ds_name in datasets:
            if col in datasets[ds_name].columns and datasets[ds_name][col].dtype == object:
                if col not in label_encoders:
                    le = LabelEncoder()
                    le.fit(datasets["train"][col].astype(str))
                    label_encoders[col] = le
                le = label_encoders[col]
                known = set(le.classes_)
                datasets[ds_name][col] = datasets[ds_name][col].astype(str).apply(
                    lambda x: le.transform([x])[0] if x in known else np.nan
                )

    for col in BINARY_VARS:
        for ds_name in datasets:
            if col in datasets[ds_name].columns and datasets[ds_name][col].dtype == object:
                datasets[ds_name][col] = datasets[ds_name][col].map({
                    True: 1, False: 0, "True": 1, "False": 0,
                    "TRUE": 1, "FALSE": 0
                })

    # Convert to numeric and drop NaN rows
    for ds_name in datasets:
        datasets[ds_name] = datasets[ds_name].apply(pd.to_numeric, errors="coerce").dropna()

    results = {}
    for ds_name, df_m in datasets.items():
        y = df_m[OUTCOME_VAR].astype(int).values
        X = df_m.drop(columns=[OUTCOME_VAR]).values
        feature_names = df_m.drop(columns=[OUTCOME_VAR]).columns.tolist()
        results[ds_name] = (X, y, feature_names)

    return results


data = encode_and_prepare(df_train, df_intval, df_tempval)
X_train, y_train, feature_names = data["train"]
X_intval, y_intval, _ = data["intval"]
X_tempval, y_tempval, _ = data["tempval"]

print(f"\n  Features: {len(feature_names)}")
print(f"  Training  : N={len(y_train):,}, SA={y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"  Internal  : N={len(y_intval):,}, SA={y_intval.sum():,} ({y_intval.mean()*100:.2f}%)")
print(f"  Temporal  : N={len(y_tempval):,}, SA={y_tempval.sum():,} ({y_tempval.mean()*100:.2f}%)")

# Standardise features (required for LR and MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_intval_scaled = scaler.transform(X_intval)
X_tempval_scaled = scaler.transform(X_tempval)

# ============================================================
# 3. Define models
# ============================================================
# Note: positive rate ~3.26% — severe class imbalance
# All models use class_weight='balanced' or scale_pos_weight
print(f"\n{'='*70}")
print("STEP 2: Initialise models")
print("=" * 70)

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"  Class imbalance ratio: 1:{spw:.1f} (positive rate {y_train.mean()*100:.2f}%)")

models = {
    "LR": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE,
        solver="lbfgs", C=1.0
    ),
    "RF": RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced",
        min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "GBM": GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, min_samples_leaf=20,
        random_state=RANDOM_STATE
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32), max_iter=500, early_stopping=True,
        validation_fraction=0.1, random_state=RANDOM_STATE,
        learning_rate="adaptive"
    ),
}

if HAS_XGB:
    models["XGB"] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, eval_metric="logloss",
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=20,
        random_state=RANDOM_STATE, n_jobs=-1, use_label_encoder=False
    )

if HAS_LGBM:
    models["LGBM"] = LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, num_leaves=31,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )

# Models that require standardised (scaled) inputs
SCALED_MODELS = {"LR", "MLP"}

# ============================================================
# 4. Train models
# ============================================================
trained_models = {}
for model_key, model in models.items():
    t0 = time.time()
    X_tr = X_train_scaled if model_key in SCALED_MODELS else X_train
    model.fit(X_tr, y_train)
    trained_models[model_key] = model
    elapsed = time.time() - t0
    print(f"  {MODEL_NAMES[model_key]:25s} trained in {elapsed:.1f}s")

# ============================================================
# 4b. Determine optimal threshold using Youden's J statistic (training-set ROC)
# ============================================================
print(f"\n  --- Optimal threshold (Youden's J statistic) ---")
optimal_thresholds = {}
for model_key, model in trained_models.items():
    X_tr = X_train_scaled if model_key in SCALED_MODELS else X_train
    y_prob_tr = model.predict_proba(X_tr)[:, 1]
    fpr_tr, tpr_tr, thresholds_tr = roc_curve(y_train, y_prob_tr)
    j_scores = tpr_tr - fpr_tr
    best_idx = np.argmax(j_scores)
    optimal_thresholds[model_key] = thresholds_tr[best_idx]
    print(f"  {MODEL_NAMES[model_key]:25s}: threshold = {thresholds_tr[best_idx]:.4f} "
          f"(Sens={tpr_tr[best_idx]:.3f}, Spec={1-fpr_tr[best_idx]:.3f})")

# ============================================================
# 5. Generate predictions
# ============================================================
print(f"\n{'='*70}")
print("STEP 3: Generating predictions")
print("=" * 70)

predictions = {}  # {model_key: {dataset_name: (y_true, y_prob, y_pred)}}

dataset_map = {
    "Training":            (X_train, X_train_scaled, y_train),
    "Internal Validation": (X_intval, X_intval_scaled, y_intval),
    "Temporal Validation": (X_tempval, X_tempval_scaled, y_tempval),
}

for model_key, model in trained_models.items():
    predictions[model_key] = {}
    thresh = optimal_thresholds[model_key]
    for ds_name, (X_raw, X_sc, y_true) in dataset_map.items():
        X_use = X_sc if model_key in SCALED_MODELS else X_raw
        y_prob = model.predict_proba(X_use)[:, 1]
        y_pred = (y_prob >= thresh).astype(int)  # apply Youden-optimal threshold
        predictions[model_key][ds_name] = (y_true, y_prob, y_pred)

print("  Predictions complete.")

# ============================================================
# 6. Bootstrap 95% CI utility
# ============================================================
def bootstrap_ci(y_true, y_score, metric_func, n_bootstrap=N_BOOTSTRAP,
                 ci=0.95, random_state=42):
    """
    Compute metric 95% CI using non-parametric bootstrap resampling.
    metric_func: function(y_true, y_score) -> scalar
    """
    rng = np.random.RandomState(random_state)
    scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        y_t = y_true[idx]
        y_s = y_score[idx]
        # Ensure both classes are represented
        if len(np.unique(y_t)) < 2:
            continue
        try:
            scores.append(metric_func(y_t, y_s))
        except:
            continue
    if len(scores) == 0:
        return np.nan, np.nan, np.nan
    scores = np.array(scores)
    alpha = 1 - ci
    lower = np.percentile(scores, alpha / 2 * 100)
    upper = np.percentile(scores, (1 - alpha / 2) * 100)
    point = metric_func(y_true, y_score)
    return point, lower, upper


def auroc_metric(y_true, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return auc(fpr, tpr)


def auprc_metric(y_true, y_prob):
    return average_precision_score(y_true, y_prob)


def brier_metric(y_true, y_prob):
    return brier_score_loss(y_true, y_prob)


def sensitivity_metric(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tp / (tp + fn) if (tp + fn) > 0 else 0


def specificity_metric(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0


def ppv_metric(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tp / (tp + fp) if (tp + fp) > 0 else 0


def npv_metric(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else 0


def f1_metric(y_true, y_pred):
    return f1_score(y_true, y_pred, zero_division=0)


def accuracy_metric(y_true, y_pred):
    return accuracy_score(y_true, y_pred)


# ============================================================
# 7. Compute all performance metrics (with 95% CI)
# ============================================================
print(f"\n{'='*70}")
print("STEP 4: Computing performance metrics (bootstrap 95% CI)")
print("=" * 70)

all_metrics = []

for model_key in trained_models:
    for ds_name in dataset_map:
        y_true, y_prob, y_pred = predictions[model_key][ds_name]
        print(f"  {MODEL_NAMES[model_key]:25s} | {ds_name:25s} ...", end="")
        t0 = time.time()

        # Probability-based metrics
        auroc_val, auroc_lo, auroc_hi = bootstrap_ci(y_true, y_prob, auroc_metric)
        auprc_val, auprc_lo, auprc_hi = bootstrap_ci(y_true, y_prob, auprc_metric)
        brier_val, brier_lo, brier_hi = bootstrap_ci(y_true, y_prob, brier_metric)

        # Classification metrics (threshold-based)
        sens_val, sens_lo, sens_hi = bootstrap_ci(y_true, y_pred, sensitivity_metric)
        spec_val, spec_lo, spec_hi = bootstrap_ci(y_true, y_pred, specificity_metric)
        ppv_val, ppv_lo, ppv_hi = bootstrap_ci(y_true, y_pred, ppv_metric)
        npv_val, npv_lo, npv_hi = bootstrap_ci(y_true, y_pred, npv_metric)
        f1_val, f1_lo, f1_hi = bootstrap_ci(y_true, y_pred, f1_metric)
        acc_val, acc_lo, acc_hi = bootstrap_ci(y_true, y_pred, accuracy_metric)

        row = {
            "Model": MODEL_NAMES[model_key],
            "Dataset": ds_name,
            "Threshold": f"{optimal_thresholds[model_key]:.4f}",
            "AUROC": f"{auroc_val:.3f} ({auroc_lo:.3f}-{auroc_hi:.3f})",
            "AUPRC": f"{auprc_val:.3f} ({auprc_lo:.3f}-{auprc_hi:.3f})",
            "Brier Score": f"{brier_val:.4f} ({brier_lo:.4f}-{brier_hi:.4f})",
            "Sensitivity": f"{sens_val:.3f} ({sens_lo:.3f}-{sens_hi:.3f})",
            "Specificity": f"{spec_val:.3f} ({spec_lo:.3f}-{spec_hi:.3f})",
            "PPV": f"{ppv_val:.3f} ({ppv_lo:.3f}-{ppv_hi:.3f})",
            "NPV": f"{npv_val:.3f} ({npv_lo:.3f}-{npv_hi:.3f})",
            "F1 Score": f"{f1_val:.3f} ({f1_lo:.3f}-{f1_hi:.3f})",
            "Accuracy": f"{acc_val:.3f} ({acc_lo:.3f}-{acc_hi:.3f})",
            # Numeric version
            "_AUROC": auroc_val,
            "_AUPRC": auprc_val,
            "_Brier": brier_val,
        }
        all_metrics.append(row)
        elapsed = time.time() - t0
        print(f" AUROC={auroc_val:.3f} ({elapsed:.0f}s)")

metrics_df = pd.DataFrame(all_metrics)

# Save performance metrics table
display_cols = [c for c in metrics_df.columns if not c.startswith("_")]
metrics_df[display_cols].to_csv(
    os.path.join(OUTPUT_DIR, "performance_metrics.csv"),
    index=False, encoding="utf-8-sig"
)
print(f"\n  Performance metrics table saved to: {OUTPUT_DIR}/performance_metrics.csv")

# Print summary
print(f"\n{'='*70}")
print("Performance metrics summary")
print("=" * 70)
for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    print(f"\n--- {ds_name} ---")
    sub = metrics_df[metrics_df["Dataset"] == ds_name][display_cols]
    print(sub.to_string(index=False))

# ============================================================
# 8. Plot AUC-ROC curves (one per dataset split)
# ============================================================
print(f"\n{'='*70}")
print("STEP 5: Plotting AUC-ROC curves")
print("=" * 70)

for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    fig, ax = plt.subplots(figsize=(7, 6.5), facecolor="white")
    ax.set_facecolor("white")

    for model_key in trained_models:
        y_true, y_prob, _ = predictions[model_key][ds_name]
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)

        # Bootstrap CI for ROC AUC
        row = [r for r in all_metrics
               if r["Model"] == MODEL_NAMES[model_key] and r["Dataset"] == ds_name][0]
        auroc_str = row["AUROC"]

        ax.plot(fpr, tpr, color=COLOR_PALETTE[model_key], linewidth=2,
                label=f"{MODEL_NAMES[model_key]} (AUC={auroc_str})")

    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("1 - Specificity (FPR)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sensitivity (TPR)", fontsize=12, fontweight="bold")
    ax.set_title(f"ROC Curve — {ds_name}", fontsize=13, fontweight="bold", pad=12)
    ax.legend(loc="lower right", fontsize=8.5, frameon=True, framealpha=0.9)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.grid(True, alpha=0.12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fname = f"roc_curve_{ds_name.replace(' ', '_').lower()}"
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.tiff"), dpi=DPI,
                bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.png"), dpi=DPI, bbox_inches="tight")
    plt.close()

print("  ROC curves saved.")

# ============================================================
# 9. Plot AUC-PR curves (one per dataset split)
# ============================================================
print(f"\n{'='*70}")
print("STEP 6: Plotting AUC-PR curves")
print("=" * 70)

for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    fig, ax = plt.subplots(figsize=(7, 6.5), facecolor="white")
    ax.set_facecolor("white")

    y_true_ds = predictions[list(trained_models.keys())[0]][ds_name][0]
    baseline = y_true_ds.mean()

    for model_key in trained_models:
        y_true, y_prob, _ = predictions[model_key][ds_name]
        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        ap = average_precision_score(y_true, y_prob)

        row = [r for r in all_metrics
               if r["Model"] == MODEL_NAMES[model_key] and r["Dataset"] == ds_name][0]
        auprc_str = row["AUPRC"]

        ax.plot(recall, precision, color=COLOR_PALETTE[model_key], linewidth=2,
                label=f"{MODEL_NAMES[model_key]} (AP={auprc_str})")

    ax.axhline(y=baseline, color="gray", linestyle="--", linewidth=0.8, alpha=0.5,
               label=f"Baseline ({baseline:.3f})")
    ax.set_xlabel("Recall (Sensitivity)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Precision (PPV)", fontsize=12, fontweight="bold")
    ax.set_title(f"Precision-Recall Curve — {ds_name}", fontsize=13, fontweight="bold", pad=12)
    ax.legend(loc="upper right", fontsize=8.5, frameon=True, framealpha=0.9)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.grid(True, alpha=0.12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fname = f"pr_curve_{ds_name.replace(' ', '_').lower()}"
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.tiff"), dpi=DPI,
                bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.png"), dpi=DPI, bbox_inches="tight")
    plt.close()

print("  PR curves saved.")

# ============================================================
# 10. Plot calibration curves
# ============================================================
print(f"\n{'='*70}")
print("STEP 7: Plotting calibration curves")
print("=" * 70)

for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    fig, ax = plt.subplots(figsize=(7, 6.5), facecolor="white")
    ax.set_facecolor("white")

    for model_key in trained_models:
        y_true, y_prob, _ = predictions[model_key][ds_name]
        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")

        # Brier Score
        row = [r for r in all_metrics
               if r["Model"] == MODEL_NAMES[model_key] and r["Dataset"] == ds_name][0]
        brier_str = row["Brier Score"]

        ax.plot(prob_pred, prob_true, "o-", color=COLOR_PALETTE[model_key], linewidth=2,
                markersize=5, label=f"{MODEL_NAMES[model_key]} (Brier={brier_str})")

    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5, label="Perfectly Calibrated")
    ax.set_xlabel("Mean Predicted Probability", fontsize=12, fontweight="bold")
    ax.set_ylabel("Observed Proportion", fontsize=12, fontweight="bold")
    ax.set_title(f"Calibration Curve — {ds_name}", fontsize=13, fontweight="bold", pad=12)
    ax.legend(loc="lower right", fontsize=8, frameon=True, framealpha=0.9)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.grid(True, alpha=0.12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fname = f"calibration_{ds_name.replace(' ', '_').lower()}"
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.tiff"), dpi=DPI,
                bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.png"), dpi=DPI, bbox_inches="tight")
    plt.close()

print("  Calibration curves saved.")

# ============================================================
# 11. Decision Curve Analysis (DCA)
# ============================================================
print(f"\n{'='*70}")
print("STEP 8: Plotting Decision Curves (DCA)")
print("=" * 70)


def calculate_net_benefit(y_true, y_prob, thresholds):
    """Compute net benefit at each probability threshold for DCA."""
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        y_pred_t = (y_prob >= thresh).astype(int)
        tp = np.sum((y_pred_t == 1) & (y_true == 1))
        fp = np.sum((y_pred_t == 1) & (y_true == 0))
        net_benefit = (tp / n) - (fp / n) * (thresh / (1 - thresh + 1e-10))
        net_benefits.append(net_benefit)
    return np.array(net_benefits)


thresholds = np.linspace(0.01, 0.50, 100)

for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")
    ax.set_facecolor("white")

    y_true_ds = predictions[list(trained_models.keys())[0]][ds_name][0]

    for model_key in trained_models:
        y_true, y_prob, _ = predictions[model_key][ds_name]
        nb = calculate_net_benefit(y_true, y_prob, thresholds)
        ax.plot(thresholds, nb, color=COLOR_PALETTE[model_key], linewidth=2,
                label=MODEL_NAMES[model_key])

    # Treat All
    prevalence = y_true_ds.mean()
    treat_all = prevalence - (1 - prevalence) * thresholds / (1 - thresholds + 1e-10)
    ax.plot(thresholds, treat_all, color="gray", linewidth=1.5, linestyle="-",
            label="Treat All")

    # Treat None
    ax.axhline(y=0, color="black", linewidth=1, linestyle="-", label="Treat None")

    ax.set_xlabel("Threshold Probability", fontsize=12, fontweight="bold")
    ax.set_ylabel("Net Benefit", fontsize=12, fontweight="bold")
    ax.set_title(f"Decision Curve Analysis — {ds_name}", fontsize=13,
                 fontweight="bold", pad=12)
    ax.legend(loc="upper right", fontsize=8.5, frameon=True, framealpha=0.9)

    y_min = min(-0.05, treat_all.min() - 0.02)
    ax.set_ylim([y_min, ax.get_ylim()[1] * 1.05])
    ax.set_xlim([0, 0.50])
    ax.grid(True, alpha=0.12)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    fname = f"dca_{ds_name.replace(' ', '_').lower()}"
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.tiff"), dpi=DPI,
                bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.png"), dpi=DPI, bbox_inches="tight")
    plt.close()

print("  DCA curves saved.")

# ============================================================
# 12. Confusion matrices
# ============================================================
print(f"\n{'='*70}")
print("STEP 9: Plotting confusion matrices")
print("=" * 70)

for ds_name in ["Training", "Internal Validation", "Temporal Validation"]:
    n_models = len(trained_models)
    fig, axes = plt.subplots(1, n_models, figsize=(4.2 * n_models, 4), facecolor="white")
    if n_models == 1:
        axes = [axes]

    for ax_i, model_key in enumerate(trained_models):
        y_true, y_prob, y_pred = predictions[model_key][ds_name]
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[ax_i],
                    xticklabels=["Negative", "Positive"],
                    yticklabels=["Negative", "Positive"],
                    cbar=False, linewidths=0.5, linecolor="white",
                    annot_kws={"size": 12, "fontweight": "bold"})
        axes[ax_i].set_xlabel("Predicted", fontsize=10, fontweight="bold")
        axes[ax_i].set_ylabel("Actual", fontsize=10, fontweight="bold")
        axes[ax_i].set_title(MODEL_NAMES[model_key], fontsize=11, fontweight="bold", pad=8)

    fig.suptitle(f"Confusion Matrix — {ds_name}", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()

    fname = f"confusion_matrix_{ds_name.replace(' ', '_').lower()}"
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.tiff"), dpi=DPI,
                bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.savefig(os.path.join(OUTPUT_DIR, f"{fname}.png"), dpi=DPI, bbox_inches="tight")
    plt.close()

print("  Confusion matrices saved.")

# ============================================================
# 13. Combined comparison heatmap (all models × all dataset splits)
# ============================================================
print(f"\n{'='*70}")
print("STEP 10: Plotting combined comparison figure")
print("=" * 70)

# AUROC heatmap
model_order = list(trained_models.keys())
ds_order = ["Training", "Internal Validation", "Temporal Validation"]

auroc_matrix = np.zeros((len(model_order), len(ds_order)))
auprc_matrix = np.zeros((len(model_order), len(ds_order)))
auroc_labels = [[""]*len(ds_order) for _ in range(len(model_order))]
auprc_labels = [[""]*len(ds_order) for _ in range(len(model_order))]

for i, mk in enumerate(model_order):
    for j, ds in enumerate(ds_order):
        row = [r for r in all_metrics
               if r["Model"] == MODEL_NAMES[mk] and r["Dataset"] == ds][0]
        auroc_matrix[i, j] = row["_AUROC"]
        auprc_matrix[i, j] = row["_AUPRC"]
        auroc_labels[i][j] = row["AUROC"]
        auprc_labels[i][j] = row["AUPRC"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5), facecolor="white")

# AUROC
sns.heatmap(auroc_matrix, annot=auroc_labels, fmt="", cmap="RdYlGn", ax=ax1,
            xticklabels=[d.replace(" ", "\n") for d in ds_order],
            yticklabels=[MODEL_NAMES[mk] for mk in model_order],
            vmin=0.5, vmax=1.0, linewidths=0.5, linecolor="white",
            annot_kws={"size": 9})
ax1.set_title("AUROC (95% CI)", fontsize=12, fontweight="bold", pad=10)

# AUPRC
sns.heatmap(auprc_matrix, annot=auprc_labels, fmt="", cmap="RdYlGn", ax=ax2,
            xticklabels=[d.replace(" ", "\n") for d in ds_order],
            yticklabels=[MODEL_NAMES[mk] for mk in model_order],
            vmin=0, vmax=1.0, linewidths=0.5, linecolor="white",
            annot_kws={"size": 9})
ax2.set_title("AUPRC (95% CI)", fontsize=12, fontweight="bold", pad=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison_heatmap.tiff"), dpi=DPI,
            bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison_heatmap.png"), dpi=DPI,
            bbox_inches="tight")
plt.close()

print("  Combined comparison figure saved.")

# ============================================================
# 14. Final summary
# ============================================================
print(f"\n{'='*70}")
print("All steps complete.")
print("=" * 70)

# Identify best model on internal validation set
best_row = metrics_df[metrics_df["Dataset"] == "Internal Validation"].sort_values(
    "_AUROC", ascending=False
).iloc[0]
print(f"\n  Best model (internal validation): {best_row['Model']}")
print(f"    AUROC = {best_row['AUROC']}")
print(f"    AUPRC = {best_row['AUPRC']}")

print(f"\n  Output files ({OUTPUT_DIR}/):")
print(f"    performance_metrics.csv              performance metrics (with 95% CI)")
print(f"    roc_curve_*.tiff/png                 ROC curves (×3 splits)")
print(f"    pr_curve_*.tiff/png                  PR curves (×3 splits)")
print(f"    calibration_*.tiff/png               calibration curves (×3 splits)")
print(f"    dca_*.tiff/png                       DCA curves (×3 splits)")
print(f"    confusion_matrix_*.tiff/png          confusion matrices (×3 splits)")
print(f"    model_comparison_heatmap.tiff/png    model comparison heatmap")

print(f"""
  Suggested methods description for manuscript:
    Six machine learning models were developed: logistic regression (LR),
    random forest (RF), gradient boosting machine (GBM), extreme gradient
    boosting (XGBoost), light gradient boosting machine (LightGBM), and
    multilayer perceptron (MLP). All models were trained on the training
    set using features selected by the Boruta algorithm. Given the
    substantial class imbalance (spontaneous abortion rate: 3.26%%),
    class weighting (balanced) was applied for LR and RF, and
    scale_pos_weight was used for XGBoost and LightGBM. Continuous
    features were standardized (z-score) for LR and MLP.

    The optimal classification threshold for each model was determined
    using Youden's J statistic (maximizing sensitivity + specificity - 1)
    on the training set, rather than using the default 0.5 threshold,
    to account for the low event rate.

    Model performance was evaluated on the training set, internal
    validation set, and temporal validation set. Discrimination was
    assessed using the area under the receiver operating characteristic
    curve (AUROC) and the area under the precision-recall curve (AUPRC),
    which is particularly informative for imbalanced outcomes. Calibration
    was evaluated using calibration plots and the Brier score. Clinical
    utility was assessed using decision curve analysis (DCA). Additional
    metrics including sensitivity, specificity, positive predictive value
    (PPV), negative predictive value (NPV), F1 score, and accuracy were
    reported at the optimal threshold. All performance metrics were
    reported with 95%% confidence intervals estimated using 1,000
    bootstrap resamples.
""")
print("Script completed.")

STEP 1: 读取数据
  训练集:     360,363
  内部验证集: 40,041
  时间验证集: 1,822

  特征数: 36
  训练集: N=360,363, SA=11,753 (3.26%)
  内部验证: N=40,041, SA=1,306 (3.26%)
  时间验证: N=1,822, SA=262 (14.38%)

STEP 2: 构建模型
  类别不平衡比: 1:29.7 (阳性率 3.26%)
  Logistic Regression       训练完成 (0.4s)
  Random Forest             训练完成 (12.0s)
  GradientBoosting          训练完成 (378.5s)
  MLP                       训练完成 (121.5s)
  XGBoost                   训练完成 (8.6s)
  LightGBM                  训练完成 (6.6s)

  --- 最优阈值 (Youden's J statistic) ---
  Logistic Regression      : threshold = 0.4865 (Sens=0.629, Spec=0.606)
  Random Forest            : threshold = 0.4658 (Sens=0.794, Spec=0.769)
  GradientBoosting         : threshold = 0.0346 (Sens=0.653, Spec=0.692)
  MLP                      : threshold = 0.0369 (Sens=0.621, Spec=0.612)
  XGBoost                  : threshold = 0.4932 (Sens=0.838, Spec=0.762)
  LightGBM                 : threshold = 0.5134 (Sens=0.750, Spec=0.746)

STEP 3: 生成预测
  预测完成。

STEP 4: 计算性能指标 (Bootstrap 95% CI)
